# Cadastros suspeitos — Emilly e Cleber

## Objetivo
Identificar cadastros potencialmente suspeitos usando dados
de clientes e endereços.

## Fontes
- Silver: ecommerce_clientes
- Silver: ecommerce_enderecos

## Etapas
1. Análise exploratória das duas tabelas.
2. Preparação das variáveis e definição do rótulo.
3. Treinamento de um modelo supervisionado.
4. Avaliação em micro-lotes não utilizados no treinamento.

## Cuidados
- Não alterar as tabelas Silver de origem.
- Documentar as variáveis e as regras usadas no rótulo.
- Não usar as flags que definem o rótulo como entradas do modelo.
- Cadastro sinalizado não significa fraude comprovada

In [0]:
# ============================================================
# BIBLIOTECAS — EDA E PREPARAÇÃO
# ============================================================

from pyspark.sql import functions as F

print("Bibliotecas carregadas.")
print("Nenhum dado foi gravado.")

In [0]:
# ============================================================
# AUTENTICAÇÃO ADLS — NOTEBOOK DE MACHINE LEARNING
# ============================================================

import os
from dotenv import load_dotenv

load_dotenv(".env")

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")

configuracoes = {
    "ADLS_CLIENT_ID": client_id,
    "ADLS_TENANT_ID": tenant_id,
    "ADLS_CLIENT_SECRET": client_secret,
    "ADLS_STORAGE_ACCOUNT_NAME": storage_account,
}

faltantes = [
    nome
    for nome, valor in configuracoes.items()
    if not valor or not valor.strip()
]

if faltantes:
    raise ValueError(
        "Configurações ausentes: "
        + ", ".join(faltantes)
        + ". Confira se o arquivo .env está acessível "
          "a partir da pasta deste notebook."
    )

host_adls = f"{storage_account}.dfs.core.windows.net"

adls_options = {
    f"fs.azure.account.auth.type.{host_adls}":
        "OAuth",

    f"fs.azure.account.oauth.provider.type.{host_adls}":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    f"fs.azure.account.oauth2.client.id.{host_adls}":
        client_id,

    f"fs.azure.account.oauth2.client.secret.{host_adls}":
        client_secret,

    f"fs.azure.account.oauth2.client.endpoint.{host_adls}":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

print("Configurações carregadas e opções do ADLS preparadas.")
print("O acesso será verificado na leitura das tabelas.")
print("Nenhum dado foi gravado.")

In [0]:
# ============================================================
# LEITURA DAS FONTES — SEM ALTERAR A SILVER
# ============================================================

base_silver = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/silver"
)

clientes_path = f"{base_silver}/ecommerce_clientes"
enderecos_path = f"{base_silver}/ecommerce_enderecos"

df_clientes_ml = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(clientes_path)
)

df_enderecos_ml = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(enderecos_path)
)

for nome, df in [
    ("CLIENTES", df_clientes_ml),
    ("ENDEREÇOS", df_enderecos_ml),
]:
    print(f"\n===== {nome} =====")
    print("Quantidade de registros:", df.count())
    print("Quantidade de colunas:", len(df.columns))
    df.printSchema()

print("\nLeitura concluída. Nenhum dado foi gravado.")

In [0]:
# ============================================================
# EDA — ESTRUTURA, NULOS E TEXTOS VAZIOS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def analisar_colunas(df, nome_tabela):
    expressoes = [F.count("*").alias("total")]

    for i, campo in enumerate(df.schema.fields):
        coluna = F.col(f"{campo.name}")

        expressoes.append(
            F.count(
                F.when(coluna.isNull(), 1)
            ).alias(f"nulos_{i}")
        )

        if isinstance(campo.dataType, StringType):
            expressoes.append(
                F.count(
                    F.when(F.trim(coluna) == "", 1)
                ).alias(f"vazios_{i}")
            )

    resultado = df.agg(*expressoes).first()
    total = resultado["total"]

    linhas = []

    for i, campo in enumerate(df.schema.fields):
        nulos = resultado[f"nulos_{i}"]

        vazios = (
            resultado[f"vazios_{i}"]
            if isinstance(campo.dataType, StringType)
            else 0
        )

        linhas.append((
            nome_tabela,
            campo.name,
            campo.dataType.simpleString(),
            total,
            nulos,
            vazios,
            round(100 * nulos / total, 2) if total else None
        ))

    return spark.createDataFrame(
        linhas,
        schema="""
            tabela string,
            coluna string,
            tipo string,
            qtd_registros long,
            qtd_nulos long,
            qtd_textos_vazios long,
            pct_nulos double
        """
    )


eda_clientes = analisar_colunas(
    df_clientes_ml,
    "ecommerce_clientes"
)

eda_enderecos = analisar_colunas(
    df_enderecos_ml,
    "ecommerce_enderecos"
)

df_eda_colunas = eda_clientes.unionByName(eda_enderecos)

display(
    df_eda_colunas.orderBy(
        "tabela",
        F.desc("qtd_nulos"),
        F.desc("qtd_textos_vazios"),
        "coluna"
    )
)

print("EDA inicial concluída.")
print("Textos vazios são contados separadamente dos nulos.")
print("Nenhum dado foi alterado ou gravado.")

In [0]:
# ============================================================
# EDA — IDENTIFICADORES E RELAÇÃO CLIENTE / ENDEREÇO
# ============================================================

from pyspark.sql import functions as F

for nome, df, obrigatorias in [
    ("clientes", df_clientes_ml, ["id_cliente"]),
    ("enderecos", df_enderecos_ml, ["id_cliente", "id_endereco"]),
]:
    faltantes = [c for c in obrigatorias if c not in df.columns]

    if faltantes:
        raise ValueError(
            f"Colunas ausentes em {nome}: {faltantes}"
        )

# Confere os tipos antes de comparar as chaves.
tipo_cliente = dict(df_clientes_ml.dtypes)["id_cliente"]
tipo_endereco = dict(df_enderecos_ml.dtypes)["id_cliente"]

if tipo_cliente != tipo_endereco:
    raise ValueError(
        "Tipos diferentes para id_cliente: "
        f"clientes={tipo_cliente}, enderecos={tipo_endereco}. "
        "Precisamos conferir antes da junção."
    )

print("===== IDENTIFICADORES =====")

for nome, df, chave in [
    ("clientes", df_clientes_ml, "id_cliente"),
    ("enderecos", df_enderecos_ml, "id_endereco"),
]:
    print(f"\nTabela: {nome}")

    display(
        df.agg(
            F.count("*").alias("qtd_registros"),
            F.countDistinct(chave).alias("ids_distintos_nao_nulos"),
            F.count(
                F.when(F.col(chave).isNull(), 1)
            ).alias("ids_nulos")
        )
    )

    repeticoes = (
        df.filter(F.col(chave).isNotNull())
        .groupBy(chave)
        .count()
        .filter(F.col("count") > 1)
    )

    display(
        repeticoes.agg(
            F.count("*").alias("ids_com_mais_de_uma_linha"),
            F.coalesce(
                F.sum(F.col("count") - 1),
                F.lit(0)
            ).alias("linhas_excedentes_por_id")
        )
    )

# IDs únicos apenas para medir cobertura.
# Os DataFrames originais permanecem intactos.
ids_clientes = (
    df_clientes_ml
    .select("id_cliente")
    .filter(F.col("id_cliente").isNotNull())
    .distinct()
)

ids_clientes_enderecos = (
    df_enderecos_ml
    .select("id_cliente")
    .filter(F.col("id_cliente").isNotNull())
    .distinct()
)

print("\n===== COBERTURA ENTRE AS TABELAS =====")

print(
    "IDs de clientes sem endereço correspondente:",
    ids_clientes.join(
        ids_clientes_enderecos,
        on="id_cliente",
        how="left_anti"
    ).count()
)

print(
    "IDs de clientes em endereços sem correspondência em clientes:",
    ids_clientes_enderecos.join(
        ids_clientes,
        on="id_cliente",
        how="left_anti"
    ).count()
)

print(
    "Linhas de endereços com id_cliente nulo:",
    df_enderecos_ml.filter(
        F.col("id_cliente").isNull()
    ).count()
)

# Distingue quantidade de linhas de quantidade de endereços.
df_enderecos_por_cliente_eda = (
    df_enderecos_ml
    .filter(F.col("id_cliente").isNotNull())
    .groupBy("id_cliente")
    .agg(
        F.count("*").alias("qtd_linhas_endereco"),
        F.countDistinct("id_endereco").alias(
            "qtd_ids_endereco_distintos"
        )
    )
)

print("\n===== DISTRIBUIÇÃO DE ENDEREÇOS POR CLIENTE =====")

display(
    df_enderecos_por_cliente_eda
    .groupBy("qtd_ids_endereco_distintos")
    .agg(
        F.count("*").alias("qtd_clientes"),
        F.sum("qtd_linhas_endereco").alias("qtd_linhas_endereco")
    )
    .orderBy("qtd_ids_endereco_distintos")
)

print("Nenhum dado foi alterado ou gravado.")

In [0]:
# ============================================================
# EDA — REPETIÇÕES DE CLIENTES
# Compara dados cadastrais sem exibir seus valores.
# ============================================================

from pyspark.sql import functions as F

campos_cadastro = [
    "uuid_cliente",
    "nome",
    "sobrenome",
    "email",
    "senha_hash",
    "dt_cadastro",
    "dt_ultima_atualizacao",
]

obrigatorias = [
    "id_cliente",
    "bronze_source_file",
    *campos_cadastro,
]

faltantes = [
    coluna
    for coluna in obrigatorias
    if coluna not in df_clientes_ml.columns
]

if faltantes:
    raise ValueError(f"Colunas ausentes: {faltantes}")

# Struct permite comparar o conjunto dos campos,
# incluindo campos que eventualmente sejam nulos.
df_repeticoes_clientes_eda = (
    df_clientes_ml
    .groupBy("id_cliente")
    .agg(
        F.count("*").alias("qtd_linhas"),
        F.countDistinct("bronze_source_file").alias(
            "qtd_arquivos_distintos"
        ),
        F.countDistinct(
            F.struct(*[F.col(c) for c in campos_cadastro])
        ).alias("qtd_conteudos_cadastrais"),
        F.min("dt_ultima_atualizacao").alias(
            "primeira_atualizacao"
        ),
        F.max("dt_ultima_atualizacao").alias(
            "ultima_atualizacao"
        ),
    )
    .filter(F.col("qtd_linhas") > 1)
    .withColumn(
        "classificacao",
        F.when(
            F.col("qtd_conteudos_cadastrais") > 1,
            "CONTEUDO_CADASTRAL_DIFERENTE"
        )
        .when(
            F.col("qtd_arquivos_distintos") > 1,
            "MESMO_CONTEUDO_EM_ARQUIVOS_DIFERENTES"
        )
        .otherwise("MESMO_CONTEUDO_NO_MESMO_ARQUIVO")
    )
)

print("===== RESUMO DAS REPETIÇÕES =====")

display(
    df_repeticoes_clientes_eda
    .groupBy("classificacao")
    .agg(
        F.count("*").alias("qtd_ids"),
        F.sum("qtd_linhas").alias("qtd_linhas"),
        F.sum(F.col("qtd_linhas") - 1).alias(
            "qtd_linhas_excedentes"
        )
    )
    .orderBy("classificacao")
)

print("===== DETALHAMENTO POR ID =====")

display(
    df_repeticoes_clientes_eda.orderBy("id_cliente")
)

print("Nenhum registro foi removido.")
print("Nenhum dado foi gravado.")

In [0]:
# EDA — CONSISTÊNCIA DAS FLAGS NOS CLIENTES REPETIDOS
from pyspark.sql import functions as F

flags_clientes = [
    f"regra_{numero:02d}_falhou"
    for numero in range(1, 11)
]

faltantes = [
    coluna for coluna in flags_clientes
    if coluna not in df_clientes_ml.columns
]

if faltantes:
    raise ValueError(f"Flags ausentes: {faltantes}")

ids_repetidos = df_repeticoes_clientes_eda.select("id_cliente")

df_flags_repetidos_eda = (
    df_clientes_ml
    .join(ids_repetidos, "id_cliente", "inner")
    .groupBy("id_cliente")
    .agg(
        F.countDistinct(
            F.struct(*[F.col(c) for c in flags_clientes])
        ).alias("qtd_combinacoes_flags")
    )
)

display(
    df_flags_repetidos_eda
    .groupBy("qtd_combinacoes_flags")
    .agg(F.count("*").alias("qtd_ids"))
    .orderBy("qtd_combinacoes_flags")
)

print("1 combinação: flags iguais entre as linhas do mesmo cliente.")
print("Mais de 1 combinação: flags divergentes; precisamos investigar.")
print("Nenhum registro foi removido ou gravado.")

In [0]:
# EDA — DISTRIBUIÇÃO DAS FLAGS DE QUALIDADE
from pyspark.sql import functions as F

def resumir_flags(df, tabela, flags):
    expressoes = [F.count("*").alias("total")]

    for flag in flags:
        expressoes.extend([
            F.sum(
                F.when(F.col(flag) == True, 1).otherwise(0)
            ).alias(f"{flag}_falhas"),
            F.sum(
                F.when(F.col(flag).isNull(), 1).otherwise(0)
            ).alias(f"{flag}_nulos")
        ])

    resumo = df.agg(*expressoes).first()
    total = resumo["total"]

    linhas = [
        (
            tabela,
            flag,
            total,
            int(resumo[f"{flag}_falhas"] or 0),
            int(resumo[f"{flag}_nulos"] or 0),
            round(
                100 * (resumo[f"{flag}_falhas"] or 0) / total, 2
            ) if total else 0.0
        )
        for flag in flags
    ]

    return spark.createDataFrame(
        linhas,
        """
        tabela string,
        regra string,
        qtd_registros long,
        qtd_falhas long,
        qtd_nulos long,
        pct_falhas double
        """
    )

flags_enderecos = sorted(
    coluna for coluna in df_enderecos_ml.columns
    if coluna.startswith("falha_r")
)

if len(flags_enderecos) != 10:
    raise ValueError(
        f"Esperadas 10 flags de endereços; encontradas "
        f"{len(flags_enderecos)}."
    )

df_eda_flags = resumir_flags(
    df_clientes_ml, "clientes", flags_clientes
).unionByName(
    resumir_flags(
        df_enderecos_ml, "enderecos", flags_enderecos
    )
)

display(
    df_eda_flags.orderBy(
        "tabela", F.desc("qtd_falhas"), "regra"
    )
)

print("Percentuais calculados sobre as linhas de cada tabela.")
print("Clientes: as repetições continuam incluídas nesta EDA.")
print("Nenhum registro foi removido ou gravado.")

In [0]:
# EDA — PREENCHIMENTO DE RUAS (LOGRADOURO)
if "logradouro" not in df_enderecos_ml.columns:
    raise ValueError("Coluna logradouro ausente em endereços.")

rua = F.trim(F.col("logradouro").cast("string"))
df_ruas_eda = df_enderecos_ml.select(
    "id_cliente", "id_endereco",
    F.when(F.col("logradouro").isNull(), "NULO")
     .when(rua == "", "VAZIO_OU_ESPACOS")
     .otherwise("PREENCHIDO").alias("situacao_logradouro"),
    F.length(rua).alias("tamanho_logradouro")
)
total_enderecos = df_enderecos_ml.count()
display(
    df_ruas_eda.groupBy("situacao_logradouro")
    .agg(F.count("*").alias("qtd_enderecos"),
         F.countDistinct("id_cliente").alias("qtd_clientes_distintos"))
    .withColumn("pct_enderecos",
        F.round(F.col("qtd_enderecos") * 100.0 / total_enderecos, 2)
        if total_enderecos else F.lit(None).cast("double"))
    .orderBy("situacao_logradouro")
)
print("Clientes distintos por situação podem se sobrepor.")
print("Preenchimento não comprova a existência da rua.")
print("Nomes de ruas não foram exibidos; nenhum dado foi gravado.")

In [0]:
# EDA — MOTIVOS DAS FALHAS DE E-MAIL
from pyspark.sql import functions as F
from pyspark.sql.window import Window

regex_email = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

df_email_eda = (
    df_clientes_ml
    .withColumn(
        "qtd_ocorrencias_email",
        F.count("*").over(Window.partitionBy("email"))
    )
    .withColumn("email_nulo", F.col("email").isNull())
    .withColumn(
        "formato_invalido",
        F.when(F.col("email").isNull(), F.lit(False))
         .otherwise(~F.col("email").rlike(regex_email))
    )
    .withColumn(
        "email_repetido",
        F.col("email").isNotNull()
        & (F.col("qtd_ocorrencias_email") > 1)
    )
)

display(
    df_email_eda
    .groupBy(
        "email_nulo",
        "formato_invalido",
        "email_repetido",
        "regra_02_falhou"
    )
    .agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct("id_cliente").alias("qtd_clientes_distintos")
    )
    .orderBy(F.desc("qtd_registros"))
)

# Diferencia repetição do mesmo cliente de e-mail
# compartilhado por IDs de clientes diferentes.
df_emails_repetidos_eda = (
    df_clientes_ml
    .filter(F.col("email").isNotNull())
    .groupBy("email")
    .agg(
        F.count("*").alias("qtd_linhas"),
        F.countDistinct("id_cliente").alias("qtd_ids_clientes")
    )
    .filter(F.col("qtd_linhas") > 1)
    .withColumn(
        "tipo_repeticao",
        F.when(
            F.col("qtd_ids_clientes") == 1,
            "MESMO CLIENTE"
        ).otherwise("CLIENTES DIFERENTES")
    )
)

display(
    df_emails_repetidos_eda
    .groupBy("tipo_repeticao")
    .agg(
        F.count("*").alias("qtd_emails"),
        F.sum("qtd_linhas").alias("qtd_registros")
    )
    .orderBy("tipo_repeticao")
)

print("Análise realizada sem alterar os e-mails.")
print("Nenhum registro foi removido ou gravado.")

In [0]:
# EDA — TAMANHO E PREENCHIMENTO DOS HASHES
from pyspark.sql import functions as F

df_hash_eda = df_clientes_ml.select(
    "id_cliente",
    "regra_04_falhou",
    F.col("senha_hash").isNull().alias("hash_nulo"),
    F.length("senha_hash").alias("tamanho_hash"),
    (
        F.length(F.trim("senha_hash")) == 0
    ).alias("hash_vazio_ou_so_espacos")
)

display(
    df_hash_eda
    .groupBy(
        "hash_nulo",
        "hash_vazio_ou_so_espacos",
        "tamanho_hash",
        "regra_04_falhou"
    )
    .agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct("id_cliente").alias("qtd_clientes_distintos")
    )
    .orderBy(F.desc("qtd_registros"))
)

print("Contagem sobre todas as linhas, incluindo repetições.")
print("O conteúdo dos hashes não foi exibido.")
print("Nenhum registro foi removido ou gravado.")

In [0]:
# ==========================================================
# EDA — DATAS DE CADASTRO, ATUALIZAÇÃO E INGESTÃO
# ==========================================================

from pyspark.sql import functions as F

# 1. Conferir as colunas necessárias
colunas_obrigatorias = [
    "id_cliente",
    "dt_cadastro",
    "dt_ultima_atualizacao",
    "bronze_ingested_at",
    "regra_05_falhou",
    "regra_06_falhou"
]

faltantes = [
    coluna
    for coluna in colunas_obrigatorias
    if coluna not in df_clientes_ml.columns
]

if faltantes:
    raise ValueError(f"Colunas ausentes: {faltantes}")

# 2. Preparar as comparações sem modificar a Silver
df_datas_eda = (
    df_clientes_ml
    .select(*colunas_obrigatorias)
    .withColumn(
        "cadastro_nulo",
        F.col("dt_cadastro").isNull()
    )
    .withColumn(
        "atualizacao_nula",
        F.col("dt_ultima_atualizacao").isNull()
    )
    .withColumn(
        "ingestao_nula",
        F.col("bronze_ingested_at").isNull()
    )
    .withColumn(
        "cadastro_apos_ingestao",
        F.col("dt_cadastro") > F.col("bronze_ingested_at")
    )
    .withColumn(
        "atualizacao_apos_ingestao",
        F.col("dt_ultima_atualizacao") > F.col("bronze_ingested_at")
    )
    .withColumn(
        "atualizacao_antes_cadastro",
        F.col("dt_ultima_atualizacao") < F.col("dt_cadastro")
    )
)

# 3. Mostrar os intervalos em texto para facilitar a leitura
intervalos = df_datas_eda.agg(
    F.min("dt_cadastro").alias("menor_data_cadastro"),
    F.max("dt_cadastro").alias("maior_data_cadastro"),
    F.min("dt_ultima_atualizacao").alias("menor_atualizacao"),
    F.max("dt_ultima_atualizacao").alias("maior_atualizacao"),
    F.min("bronze_ingested_at").alias("primeira_ingestao"),
    F.max("bronze_ingested_at").alias("ultima_ingestao")
).first()

print("===== INTERVALOS DAS DATAS =====")
print("Fuso da sessão:", spark.conf.get("spark.sql.session.timeZone"))

for campo, valor in intervalos.asDict().items():
    print(f"{campo}: {valor}")

# 4. Contar cada situação
def contar_condicao(condicao, nome):
    return F.coalesce(
        F.sum(F.when(condicao, 1).otherwise(0)),
        F.lit(0)
    ).alias(nome)


resumo_datas = df_datas_eda.agg(
    F.count("*").alias("total_registros"),

    contar_condicao(
        F.col("cadastro_nulo"),
        "cadastros_nulos"
    ),
    contar_condicao(
        F.col("atualizacao_nula"),
        "atualizacoes_nulas"
    ),
    contar_condicao(
        F.col("ingestao_nula"),
        "ingestoes_nulas"
    ),
    contar_condicao(
        F.col("cadastro_apos_ingestao"),
        "cadastros_apos_ingestao"
    ),
    contar_condicao(
        F.col("atualizacao_apos_ingestao"),
        "atualizacoes_apos_ingestao"
    ),
    contar_condicao(
        F.col("atualizacao_antes_cadastro"),
        "atualizacoes_antes_cadastro"
    ),
    contar_condicao(
        F.col("cadastro_apos_ingestao").isNull(),
        "comparacoes_cadastro_ingestao_sem_resultado"
    ),
    contar_condicao(
        F.col("atualizacao_apos_ingestao").isNull(),
        "comparacoes_atualizacao_ingestao_sem_resultado"
    ),
    contar_condicao(
        F.col("regra_05_falhou") == True,
        "falhas_regra_05_na_silver"
    ),
    contar_condicao(
        F.col("regra_06_falhou") == True,
        "falhas_regra_06_na_silver"
    ),
    contar_condicao(
        F.col("cadastro_apos_ingestao")
        & (F.col("regra_05_falhou") == False),
        "cadastros_apos_ingestao_com_regra05_false"
    )
).first()

print("\n===== RESUMO DAS CONTAGENS =====")

for campo, valor in resumo_datas.asDict().items():
    print(f"{campo}: {valor}")

# 5. Comparar a análise temporal com as flags existentes
print("\n===== COMPARAÇÃO COM AS FLAGS DA SILVER =====")

display(
    df_datas_eda
    .groupBy(
        "cadastro_apos_ingestao",
        "atualizacao_apos_ingestao",
        "atualizacao_antes_cadastro",
        "regra_05_falhou",
        "regra_06_falhou"
    )
    .agg(F.count("*").alias("qtd_registros"))
    .orderBy(F.desc("qtd_registros"))
)

print("\nReferência temporal desta EDA: bronze_ingested_at.")
print("Comparações nulas indicam ausência de alguma data necessária.")
print("As contagens incluem as linhas repetidas.")
print("Diferenças em relação às flags exigem conferir seus critérios.")
print("Nenhum registro foi removido, alterado ou gravado.")

In [0]:
# ==========================================================
# EDA — DISTRIBUIÇÃO DOS DOMÍNIOS DE E-MAIL
# ==========================================================

from pyspark.sql import functions as F

regex_email = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

df_dominios_eda = (
    df_clientes_ml
    .select(
        "id_cliente",
        "email",
        "regra_09_falhou"
    )
    .withColumn(
        "dominio_email",
        F.when(
            F.col("email").isNull(),
            F.lit("EMAIL_NULO")
        )
        .when(
            ~F.col("email").rlike(regex_email),
            F.lit("FORMATO_INVALIDO")
        )
        .otherwise(
            F.lower(
                F.regexp_extract("email", r"@([^@]+)$", 1)
            )
        )
    )
)

total_registros = df_clientes_ml.count()

df_resumo_dominios_eda = (
    df_dominios_eda
    .groupBy("dominio_email", "regra_09_falhou")
    .agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct("id_cliente").alias("qtd_clientes_distintos")
    )
    .withColumn(
        "pct_registros",
        F.round(
            F.col("qtd_registros")
            / F.lit(total_registros) * 100,
            2
        ) if total_registros else F.lit(0.0)
    )
    .orderBy(F.desc("qtd_registros"), "dominio_email")
)

display(df_resumo_dominios_eda)

print("Percentuais calculados sobre todas as linhas de clientes.")
print("E-mails completos não foram exibidos.")
print("Domínio não reconhecido, sozinho, não comprova fraude.")
print("Nenhum registro foi removido, alterado ou gravado.")

## Interpretação da EDA
As contagens acima incluem repetições de clientes. Flags da Silver são resultados de regras existentes, não comprovação de fraude. A visão geral de colunas e flags continua cobrindo as fontes, mas CEP não será usado nos atributos desta versão.

In [0]:
# ==========================================================
# ENGENHARIA DE ATRIBUTOS DOS CLIENTES
# Não altera nem grava dados na Silver.
# ==========================================================

from pyspark.sql import functions as F

# ----------------------------------------------------------
# 1. Validar as colunas necessárias
# ----------------------------------------------------------

colunas_necessarias = [
    "id_cliente",
    "email",
    "senha_hash",
    "dt_cadastro",
    "bronze_ingested_at",
    "bronze_source_file"
]

colunas_faltantes = [
    coluna
    for coluna in colunas_necessarias
    if coluna not in df_clientes_ml.columns
]

if colunas_faltantes:
    raise ValueError(
        f"Colunas ausentes na base de clientes: {colunas_faltantes}"
    )

regex_email = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

# ----------------------------------------------------------
# 2. Criar os atributos em um novo DataFrame
# ----------------------------------------------------------

df_features_clientes_eda = (
    df_clientes_ml

    .withColumn(
        "_email_formato_valido",
        F.coalesce(
            F.col("email").rlike(regex_email),
            F.lit(False)
        )
    )

    .withColumn(
        "_usuario_email",
        F.when(
            F.col("_email_formato_valido"),
            F.regexp_extract("email", r"^([^@]+)@", 1)
        )
    )

    .withColumn(
        "dominio_email",
        F.when(
            F.col("_email_formato_valido"),
            F.lower(
                F.regexp_extract("email", r"@([^@]+)$", 1)
            )
        )
    )

    .withColumn(
        "tamanho_usuario_email",
        F.length("_usuario_email")
    )

    .withColumn(
        "qtd_digitos_email",
        F.length(
            F.regexp_replace("_usuario_email", r"[^0-9]", "")
        )
    )

    .withColumn(
        "tamanho_hash",
        F.length("senha_hash")
    )

    # Diferença em dias, incluindo frações de dia.
    # Idade negativa é preservada para análise.
    .withColumn(
        "idade_cadastro_dias",
        (
            F.col("bronze_ingested_at").cast("double")
            - F.col("dt_cadastro").cast("double")
        ) / F.lit(86400.0)
    )

    .select(
        # Identificação e rastreabilidade: não são entradas preditivas.
        "id_cliente",
        "bronze_source_file",
        "bronze_ingested_at",

        # Atributos candidatos para o modelo.
        "dominio_email",
        "tamanho_usuario_email",
        "qtd_digitos_email",
        "tamanho_hash",
        "idade_cadastro_dias"
    )
)

# ----------------------------------------------------------
# 3. Calcular contagens e validações
# ----------------------------------------------------------

atributos_numericos = [
    "tamanho_usuario_email",
    "qtd_digitos_email",
    "tamanho_hash",
    "idade_cadastro_dias"
]

atributos = ["dominio_email"] + atributos_numericos


def contar_condicao(condicao, nome):
    return F.coalesce(
        F.sum(
            F.when(condicao, 1).otherwise(0)
        ),
        F.lit(0)
    ).alias(nome)


total_origem = df_clientes_ml.count()

resumo_validacao = (
    df_features_clientes_eda
    .agg(
        F.count("*").alias("total_registros"),

        F.countDistinct("id_cliente").alias(
            "clientes_distintos"
        ),

        contar_condicao(
            F.col("id_cliente").isNull(),
            "id_cliente_nulos"
        ),

        *[
            contar_condicao(
                F.col(coluna).isNull(),
                f"{coluna}_nulos"
            )
            for coluna in atributos
        ],

        contar_condicao(
            F.col("idade_cadastro_dias") < 0,
            "idades_negativas"
        )
    )
    .first()
    .asDict()
)

total_features = resumo_validacao["total_registros"]

if total_origem != total_features:
    raise ValueError(
        "A quantidade de linhas mudou durante a criação "
        f"dos atributos: {total_origem} -> {total_features}."
    )

# ----------------------------------------------------------
# 4. Exibir contagens
# ----------------------------------------------------------

print("===== BASE DE ATRIBUTOS DOS CLIENTES =====")
print(f"Registros de origem: {total_origem}")
print(f"Registros após criar atributos: {total_features}")
print(
    f"Clientes distintos: "
    f"{resumo_validacao['clientes_distintos']}"
)
print(
    f"IDs de cliente nulos: "
    f"{resumo_validacao['id_cliente_nulos']}"
)

# ----------------------------------------------------------
# 5. Exibir resumo dos atributos numéricos
# ----------------------------------------------------------

print("\n===== RESUMO DOS ATRIBUTOS NUMÉRICOS =====")

display(
    df_features_clientes_eda
    .select(*atributos_numericos)
    .summary(
        "count",
        "min",
        "25%",
        "50%",
        "75%",
        "max"
    )
)

# ----------------------------------------------------------
# 6. Exibir validação em texto, sem rolagem horizontal
# ----------------------------------------------------------

print("\n===== NULOS E CONSISTÊNCIA TEMPORAL =====")

for coluna in atributos:
    nome_indicador = f"{coluna}_nulos"
    print(
        f"{nome_indicador}: "
        f"{resumo_validacao[nome_indicador]}"
    )

print(
    f"idades_negativas: "
    f"{resumo_validacao['idades_negativas']}"
)

# ----------------------------------------------------------
# 7. Observações
# ----------------------------------------------------------

print("\n===== OBSERVAÇÕES =====")
print("Idade negativa significa cadastro posterior à ingestão.")
print("Esses valores foram preservados para análise.")
print("As linhas repetidas ainda estão presentes.")
print("E-mails completos e conteúdo dos hashes não foram exibidos.")
print("Os atributos ainda serão avaliados antes do treinamento.")
print("Nenhum rótulo foi criado e nenhum dado foi gravado.")

In [0]:
# ============================================================
#  DISPONIBILIDADE TEMPORAL DOS ENDEREÇOS
# ============================================================

from pyspark.sql import functions as F

# 1. Validar as colunas necessárias
colunas_clientes = ["id_cliente", "bronze_ingested_at"]
colunas_enderecos = [
    "id_cliente",
    "id_endereco",
    "bronze_ingested_at"
]

for nome, df, colunas in [
    ("clientes", df_clientes_ml, colunas_clientes),
    ("enderecos", df_enderecos_ml, colunas_enderecos)
]:
    faltantes = [c for c in colunas if c not in df.columns]

    if faltantes:
        raise ValueError(
            f"Colunas ausentes em {nome}: {faltantes}"
        )

    tipo_data = dict(df.dtypes)["bronze_ingested_at"]

    if tipo_data not in ("timestamp", "timestamp_ntz"):
        raise ValueError(
            f"bronze_ingested_at em {nome} está como "
            f"{tipo_data}. É necessário conferir o tipo "
            "antes de comparar as datas."
        )


# 2. Criar referências distintas de cliente e instante de ingestão
# Isso evita multiplicar a análise por cópias da mesma referência.
# Os DataFrames originais permanecem intactos.
df_referencias_clientes_eda = (
    df_clientes_ml
    .select(
        "id_cliente",
        F.col("bronze_ingested_at").alias("instante_referencia")
    )
    .distinct()
)

df_tempos_enderecos_eda = (
    df_enderecos_ml
    .select(
        "id_cliente",
        "id_endereco",
        F.col("bronze_ingested_at").alias("ingestao_endereco"),
        F.lit(1).alias("_endereco_encontrado")
    )
)


# 3. Comparar os endereços com cada referência do cliente
df_comparacao_temporal_eda = (
    df_referencias_clientes_eda.alias("c")
    .join(
        df_tempos_enderecos_eda.alias("e"),
        F.col("c.id_cliente") == F.col("e.id_cliente"),
        "left"
    )
    .select(
        F.col("c.id_cliente").alias("id_cliente"),
        F.col("c.instante_referencia"),
        F.col("e.id_endereco"),
        F.col("e.ingestao_endereco"),
        F.col("e._endereco_encontrado")
    )
    .withColumn(
        "situacao_temporal",
        F.when(
            F.col("_endereco_encontrado").isNull(),
            F.lit("SEM_ENDERECO_CORRESPONDENTE")
        )
        .when(
            F.col("instante_referencia").isNull()
            | F.col("ingestao_endereco").isNull(),
            F.lit("SEM_DATA_PARA_COMPARAR")
        )
        .when(
            F.col("ingestao_endereco")
            <= F.col("instante_referencia"),
            F.lit("INGESTAO_ATE_A_REFERENCIA")
        )
        .otherwise(
            F.lit("INGESTAO_APOS_A_REFERENCIA")
        )
    )
)


# 4. Resumir a disponibilidade por referência de cliente
df_disponibilidade_enderecos_eda = (
    df_comparacao_temporal_eda
    .groupBy("id_cliente", "instante_referencia")
    .agg(
        F.sum(
            F.when(
                F.col("_endereco_encontrado").isNotNull(), 1
            ).otherwise(0)
        ).alias("qtd_linhas_endereco"),

        F.sum(
            F.when(
                F.col("situacao_temporal")
                == "INGESTAO_ATE_A_REFERENCIA", 1
            ).otherwise(0)
        ).alias("qtd_enderecos_ate_referencia"),

        F.sum(
            F.when(
                F.col("situacao_temporal")
                == "INGESTAO_APOS_A_REFERENCIA", 1
            ).otherwise(0)
        ).alias("qtd_enderecos_apos_referencia"),

        F.sum(
            F.when(
                F.col("situacao_temporal")
                == "SEM_DATA_PARA_COMPARAR", 1
            ).otherwise(0)
        ).alias("qtd_enderecos_sem_comparacao")
    )
)


# 5. Mostrar contagens gerais
print("===== REFERÊNCIAS ANALISADAS =====")
print(
    "Linhas de clientes na origem:",
    df_clientes_ml.count()
)
print(
    "Referências distintas de cliente e ingestão:",
    df_referencias_clientes_eda.count()
)
print(
    "Linhas de endereços na origem:",
    df_enderecos_ml.count()
)

print("\n===== COMPARAÇÕES TEMPORAIS =====")

display(
    df_comparacao_temporal_eda
    .groupBy("situacao_temporal")
    .agg(
        F.count("*").alias("qtd_comparacoes"),
        F.countDistinct("id_cliente").alias(
            "qtd_clientes_distintos"
        )
    )
    .orderBy("situacao_temporal")
)


# 6. Mostrar quantas referências têm endereços disponíveis
print("\n===== DISPONIBILIDADE POR REFERÊNCIA =====")

display(
    df_disponibilidade_enderecos_eda
    .select(
        (F.col("qtd_linhas_endereco") > 0).alias(
            "tem_endereco_correspondente"
        ),
        (F.col("qtd_enderecos_ate_referencia") > 0).alias(
            "tem_endereco_ate_referencia"
        ),
        (F.col("qtd_enderecos_apos_referencia") > 0).alias(
            "tem_endereco_apos_referencia"
        ),
        (F.col("qtd_enderecos_sem_comparacao") > 0).alias(
            "tem_endereco_sem_comparacao"
        )
    )
    .groupBy(
        "tem_endereco_correspondente",
        "tem_endereco_ate_referencia",
        "tem_endereco_apos_referencia",
        "tem_endereco_sem_comparacao"
    )
    .count()
    .withColumnRenamed("count", "qtd_referencias")
    .orderBy(F.desc("qtd_referencias"))
)

print("\n===== OBSERVAÇÕES =====")
print("Referência utilizada: ingestão de cada registro de cliente.")
print("Um cliente pode ter mais de um instante de referência.")
print("Um endereço pode aparecer em mais de uma comparação.")
print("Clientes distintos por situação não devem ser somados.")
print("Ingestão posterior não significa que o endereço seja inválido.")
print(
    "Esta análise confere a ingestão; não comprova que a Silver "
    "preserve todas as versões históricas dos endereços."
)
print("Nenhum atributo de endereço foi incorporado ao modelo.")
print("Nenhum registro de origem foi alterado, removido ou gravado.")

In [0]:
# ============================================================
# ATRIBUTOS DOS ENDEREÇOS POR REFERÊNCIA
# Rua, latitude e longitude — sem atributos de CEP
# ============================================================

from pyspark.sql import functions as F

# 1. Validar as colunas utilizadas
necessarias = [
    "id_cliente",
    "id_endereco",
    "bronze_ingested_at",
    "logradouro",
    "latitude",
    "longitude"
]

faltantes = [
    coluna for coluna in necessarias
    if coluna not in df_enderecos_ml.columns
]

if faltantes:
    raise ValueError(
        f"Colunas ausentes em endereços: {faltantes}"
    )

# A agregação pressupõe uma linha por endereço.
validacao_ids = df_enderecos_ml.agg(
    F.count("*").alias("linhas"),
    F.count("id_endereco").alias("ids_preenchidos"),
    F.countDistinct("id_endereco").alias("ids_distintos")
).first()

if not (
    validacao_ids["linhas"]
    == validacao_ids["ids_preenchidos"]
    == validacao_ids["ids_distintos"]
):
    raise ValueError(
        "Existem IDs de endereço nulos ou repetidos. "
        "Revisar antes de calcular os atributos."
    )

# Validar as referências para evitar multiplicação na junção.
chaves_referencia = ["id_cliente", "instante_referencia"]

faltantes_referencia = [
    coluna for coluna in chaves_referencia
    if coluna not in df_referencias_clientes_eda.columns
]

if faltantes_referencia:
    raise ValueError(
        f"Colunas ausentes nas referências: {faltantes_referencia}"
    )

if (
    df_referencias_clientes_eda
    .filter(
        F.col("id_cliente").isNull()
        | F.col("instante_referencia").isNull()
    )
    .limit(1)
    .count() > 0
):
    raise ValueError("Existem chaves nulas nas referências.")

if (
    df_referencias_clientes_eda
    .groupBy(*chaves_referencia)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count() > 0
):
    raise ValueError("Existem referências repetidas.")

# 2. Preparar os campos em um novo DataFrame
# Os valores originais da Silver permanecem intactos.
df_enderecos_preparados_eda = (
    df_enderecos_ml
    .select(
        "id_cliente",
        "id_endereco",
        F.col("bronze_ingested_at").alias("ingestao_endereco"),
        F.trim(F.col("logradouro").cast("string")).alias(
            "_logradouro_texto"
        ),
        F.trim(F.col("latitude").cast("string")).alias(
            "_latitude_texto"
        ),
        F.trim(F.col("longitude").cast("string")).alias(
            "_longitude_texto"
        )
    )
    # Valores não conversíveis ficam nulos nas colunas auxiliares.
    .withColumn(
        "_latitude_num",
        F.expr("try_cast(_latitude_texto AS DOUBLE)")
    )
    .withColumn(
        "_longitude_num",
        F.expr("try_cast(_longitude_texto AS DOUBLE)")
    )
)

logradouro_ausente = (
    F.col("_logradouro_texto").isNull()
    | (F.col("_logradouro_texto") == "")
)

latitude_ausente = (
    F.col("_latitude_texto").isNull()
    | (F.col("_latitude_texto") == "")
)

longitude_ausente = (
    F.col("_longitude_texto").isNull()
    | (F.col("_longitude_texto") == "")
)

# Ausência é analisada separadamente da validade do valor preenchido.
latitude_invalida = (
    ~latitude_ausente
    & (
        F.col("_latitude_num").isNull()
        | F.isnan("_latitude_num")
        | ~F.col("_latitude_num").between(-90.0, 90.0)
    )
)

longitude_invalida = (
    ~longitude_ausente
    & (
        F.col("_longitude_num").isNull()
        | F.isnan("_longitude_num")
        | ~F.col("_longitude_num").between(-180.0, 180.0)
    )
)

df_enderecos_atributos_eda = (
    df_enderecos_preparados_eda
    .select(
        "id_cliente",
        "id_endereco",
        "ingestao_endereco",
        logradouro_ausente.cast("int").alias(
            "_logradouro_ausente"
        ),
        (
            latitude_ausente | longitude_ausente
        ).cast("int").alias("_coordenadas_ausentes"),
        (
            latitude_invalida | longitude_invalida
        ).cast("int").alias("_coordenadas_invalidas")
    )
)

# 3. Considerar apenas endereços ingeridos até a referência
# A classificação ocorre antes do left join:
# referências sem endereço terão indicadores nulos.
df_enderecos_no_instante_eda = (
    df_referencias_clientes_eda.alias("c")
    .join(
        df_enderecos_atributos_eda.alias("e"),
        (
            F.col("c.id_cliente") == F.col("e.id_cliente")
        )
        & (
            F.col("e.ingestao_endereco")
            <= F.col("c.instante_referencia")
        ),
        "left"
    )
    .select(
        F.col("c.id_cliente").alias("id_cliente"),
        F.col("c.instante_referencia"),
        F.col("e.id_endereco"),
        F.col("e._logradouro_ausente"),
        F.col("e._coordenadas_ausentes"),
        F.col("e._coordenadas_invalidas")
    )
)

# 4. Agregar por cliente e referência
# Denominador: todos os endereços disponíveis naquela referência.
df_features_enderecos_eda = (
    df_enderecos_no_instante_eda
    .groupBy(*chaves_referencia)
    .agg(
        F.count("id_endereco").alias(
            "qtd_enderecos_disponiveis"
        ),
        F.avg("_logradouro_ausente").alias(
            "proporcao_logradouros_ausentes"
        ),
        F.avg("_coordenadas_ausentes").alias(
            "proporcao_coordenadas_ausentes"
        ),
        F.avg("_coordenadas_invalidas").alias(
            "proporcao_coordenadas_invalidas"
        )
    )
    .withColumn(
        "tem_endereco_disponivel",
        F.col("qtd_enderecos_disponiveis") > 0
    )
)

# 5. Validar a preservação das referências
total_referencias = df_referencias_clientes_eda.count()
total_resultado = df_features_enderecos_eda.count()

if total_resultado != total_referencias:
    raise ValueError(
        "A quantidade de referências mudou durante a agregação."
    )

proporcoes_enderecos = [
    "proporcao_logradouros_ausentes",
    "proporcao_coordenadas_ausentes",
    "proporcao_coordenadas_invalidas"
]

for coluna in proporcoes_enderecos:
    inconsistencias = (
        df_features_enderecos_eda
        .filter(
            (
                F.col("tem_endereco_disponivel")
                & (
                    F.col(coluna).isNull()
                    | ~F.col(coluna).between(0.0, 1.0)
                )
            )
            | (
                ~F.col("tem_endereco_disponivel")
                & F.col(coluna).isNotNull()
            )
        )
        .limit(1)
        .count()
    )

    if inconsistencias:
        raise ValueError(
            f"Proporção inconsistente: {coluna}"
        )

# 6. Exibir resultados
print("===== ATRIBUTOS DE ENDEREÇOS =====")
print("Referências de origem:", total_referencias)
print("Referências após a agregação:", total_resultado)

print("\n===== DISPONIBILIDADE =====")

display(
    df_features_enderecos_eda
    .groupBy("tem_endereco_disponivel")
    .agg(
        F.count("*").alias("qtd_referencias"),
        F.sum("qtd_enderecos_disponiveis").alias(
            "total_enderecos_disponiveis"
        )
    )
    .orderBy("tem_endereco_disponivel")
)

print("\n===== RESUMO ENTRE REFERÊNCIAS COM ENDEREÇO =====")

display(
    df_features_enderecos_eda
    .filter(F.col("tem_endereco_disponivel"))
    .select(
        "qtd_enderecos_disponiveis",
        *proporcoes_enderecos
    )
    .summary("count", "min", "25%", "50%", "75%", "max")
)

print("\n===== NULOS NAS PROPORÇÕES =====")

resumo_nulos = (
    df_features_enderecos_eda
    .agg(*[
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(coluna)
        for coluna in proporcoes_enderecos
    ])
    .first()
    .asDict()
)

for coluna, quantidade in resumo_nulos.items():
    print(f"{coluna}: {quantidade}")

print("\n===== OBSERVAÇÕES =====")
print("As proporções variam de 0 a 1; 0,5 representa 50%.")
print("Sem endereço disponível, as proporções permanecem nulas.")
print("Coordenadas ausentes e inválidas podem ocorrer no mesmo endereço.")
print("Coordenada zero, isoladamente, não é considerada inválida.")
print("Faixas válidas não comprovam a localização correta do endereço.")
print("A análise depende das versões presentes na Silver consultada.")
print("Esta célula não cria nem modifica o rótulo experimental.")
print("Nenhum dado de origem foi alterado ou gravado.")

### Atributos de endereços disponíveis na referência

A agregação preservou as 1.135 referências de cliente e ingestão.

Foram considerados 323 endereços disponíveis para 296 referências.
As outras 839 referências não tinham endereço ingerido até aquele
instante; suas proporções permaneceram nulas.

Entre as referências com endereço disponível:
- A proporção de logradouros ausentes variou de 0 a 1,
  com mediana zero.
- As proporções de coordenadas ausentes e inválidas foram zero
  em todas as 296 referências, conforme os critérios implementados.

Coordenadas dentro das faixas permitidas não comprovam que a
localização esteja correta. Logradouro preenchido não comprova
a existência da rua.

Nenhum dado da Silver foi alterado.

In [0]:
# ============================================================
# UNIÃO DOS ATRIBUTOS DE CLIENTES E ENDEREÇOS
# ============================================================

from pyspark.sql import functions as F

chaves = ["id_cliente", "bronze_ingested_at"]

atributos_clientes = [
    "dominio_email",
    "tamanho_usuario_email",
    "qtd_digitos_email",
    "tamanho_hash",
    "idade_cadastro_dias"
]

atributos_enderecos = [
    "qtd_enderecos_disponiveis",
    "tem_endereco_disponivel",
    "proporcao_logradouros_ausentes",
    "proporcao_coordenadas_ausentes",
    "proporcao_coordenadas_invalidas"
]

proporcoes_enderecos = [
    "proporcao_logradouros_ausentes",
    "proporcao_coordenadas_ausentes",
    "proporcao_coordenadas_invalidas"
]

# 1. Validar as colunas antes da junção
for nome, df, obrigatorias in [
    (
        "atributos de clientes",
        df_features_clientes_eda,
        chaves + atributos_clientes
    ),
    (
        "atributos de endereços",
        df_features_enderecos_eda,
        ["id_cliente", "instante_referencia"] + atributos_enderecos
    )
]:
    faltantes = [
        coluna for coluna in obrigatorias
        if coluna not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"Colunas ausentes em {nome}: {faltantes}"
        )

# 2. Padronizar a referência dos endereços
df_enderecos_para_join_eda = (
    df_features_enderecos_eda
    .withColumnRenamed(
        "instante_referencia",
        "bronze_ingested_at"
    )
    .select(*chaves, *atributos_enderecos)
    .withColumn("_referencia_encontrada", F.lit(True))
)

# 3. Validar as chaves
for nome, df in [
    ("clientes", df_features_clientes_eda),
    ("endereços agregados", df_enderecos_para_join_eda)
]:
    if (
        df.filter(
            F.col("id_cliente").isNull()
            | F.col("bronze_ingested_at").isNull()
        )
        .limit(1)
        .count() > 0
    ):
        raise ValueError(
            f"Existem chaves nulas em {nome}. "
            "Revisar antes da junção."
        )

tipos_clientes = dict(df_features_clientes_eda.dtypes)
tipos_enderecos = dict(df_enderecos_para_join_eda.dtypes)

for chave in chaves:
    if tipos_clientes[chave] != tipos_enderecos[chave]:
        raise ValueError(
            f"Tipos diferentes para {chave}: "
            f"clientes={tipos_clientes[chave]}, "
            f"endereços={tipos_enderecos[chave]}."
        )

# A base agregada deve ter somente uma linha por referência.
if (
    df_enderecos_para_join_eda
    .groupBy(*chaves)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "Há referências repetidas nos atributos de endereços. "
        "A junção poderia multiplicar as linhas de clientes."
    )

conflitos = [
    coluna
    for coluna in atributos_enderecos + ["_referencia_encontrada"]
    if coluna in df_features_clientes_eda.columns
]

if conflitos:
    raise ValueError(
        f"A base de clientes já contém estas colunas: {conflitos}"
    )

# 4. Unir mantendo todas as colunas da base de clientes
df_features_unidas_eda = (
    df_features_clientes_eda
    .join(
        df_enderecos_para_join_eda,
        on=chaves,
        how="left"
    )
)

# 5. Conferir a preservação das linhas
total_antes = df_features_clientes_eda.count()
total_depois = df_features_unidas_eda.count()

sem_correspondencia = (
    df_features_unidas_eda
    .filter(F.col("_referencia_encontrada").isNull())
    .count()
)

if total_antes != total_depois:
    raise ValueError(
        "A junção alterou a quantidade de linhas: "
        f"{total_antes} -> {total_depois}."
    )

if sem_correspondencia > 0:
    raise ValueError(
        f"{sem_correspondencia} linhas não encontraram sua "
        "referência na base agregada de endereços. "
        "Isso é diferente de não ter endereço disponível."
    )

df_features_unidas_eda = (
    df_features_unidas_eda
    .drop("_referencia_encontrada")
)

# 6. Conferir proporções e disponibilidade
# Sem endereço: quantidade zero e proporções nulas.
# Com endereço: proporções preenchidas entre zero e um.
disponivel = F.col("tem_endereco_disponivel")
quantidade = F.col("qtd_enderecos_disponiveis")

condicao_inconsistente = (
    disponivel.isNull()
    | quantidade.isNull()
    | (quantidade < 0)
    | (disponivel & (quantidade == 0))
    | (~disponivel & (quantidade != 0))
)

for coluna in proporcoes_enderecos:
    proporcao = F.col(coluna)

    condicao_inconsistente = (
        condicao_inconsistente
        | (
            disponivel
            & (
                proporcao.isNull()
                | F.isnan(proporcao)
                | ~proporcao.between(0.0, 1.0)
            )
        )
        | (
            ~disponivel
            & proporcao.isNotNull()
        )
    )

if (
    df_features_unidas_eda
    .filter(condicao_inconsistente)
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "Há inconsistências entre disponibilidade, "
        "quantidade de endereços e proporções."
    )

# 7. Exibir validações gerais
total_referencias = (
    df_features_unidas_eda
    .select(*chaves)
    .distinct()
    .count()
)

total_clientes = (
    df_features_unidas_eda
    .select("id_cliente")
    .distinct()
    .count()
)

print("===== VALIDAÇÃO DA JUNÇÃO =====")
print("Linhas antes:", total_antes)
print("Linhas depois:", total_depois)
print("Referências distintas:", total_referencias)
print("Clientes distintos:", total_clientes)
print(
    "Linhas sem correspondência na agregação:",
    sem_correspondencia
)

print("\n===== DISPONIBILIDADE NA BASE UNIDA =====")

display(
    df_features_unidas_eda
    .groupBy("tem_endereco_disponivel")
    .agg(
        F.count("*").alias("qtd_linhas"),
        F.countDistinct("id_cliente").alias(
            "qtd_clientes_distintos"
        )
    )
    .orderBy("tem_endereco_disponivel")
)

# 8. Conferir nulos sem preencher valores automaticamente
atributos_candidatos = (
    atributos_clientes + atributos_enderecos
)

resumo_nulos = (
    df_features_unidas_eda
    .agg(*[
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(coluna)
        for coluna in atributos_candidatos
    ])
    .first()
    .asDict()
)

print("\n===== NULOS POR ATRIBUTO =====")

for coluna in atributos_candidatos:
    print(f"{coluna}: {resumo_nulos[coluna]}")

print("\n===== ATRIBUTOS PRESERVADOS NA BASE =====")

for coluna in atributos_candidatos:
    print(f"- {coluna}")

print("\n===== OBSERVAÇÕES =====")
print("Domínio, tamanho do hash e idade do cadastro foram preservados.")
print("Rua e coordenadas foram incorporadas por referência temporal.")
print("As cópias de clientes ainda permanecem nesta etapa.")
print("Proporções nulas por falta de endereço foram preservadas.")
print("Os nulos são contados por linha, incluindo cópias.")
print("IDs, arquivo de origem e ingestão são metadados de controle.")
print("Esta célula não cria nem modifica o rótulo experimental.")
print("Nenhum modelo foi treinado e nenhum dado foi gravado.")

In [0]:
# ============================================================
# BASE DE ATRIBUTOS SEM CÓPIAS IDÊNTICAS
# ============================================================

from pyspark.sql import functions as F

proporcoes = [
    "proporcao_logradouros_ausentes",
    "proporcao_coordenadas_ausentes",
    "proporcao_coordenadas_invalidas"
]

atributos_clientes = [
    "dominio_email",
    "tamanho_usuario_email",
    "qtd_digitos_email",
    "tamanho_hash",
    "idade_cadastro_dias"
]

obrigatorias = [
    "id_cliente",
    "bronze_ingested_at",
    "qtd_enderecos_disponiveis",
    "tem_endereco_disponivel",
    *atributos_clientes,
    *proporcoes
]

faltantes = [
    coluna for coluna in obrigatorias
    if coluna not in df_features_unidas_eda.columns
]

if faltantes:
    raise ValueError(
        f"Colunas ausentes na base unida: {faltantes}"
    )

# 1. Consolidar somente linhas iguais em todas as colunas
# da base de atributos, incluindo os metadados presentes.
df_features_sem_copias_eda = (
    df_features_unidas_eda.distinct()
)

total_antes = df_features_unidas_eda.count()
total_depois = df_features_sem_copias_eda.count()

print("===== CONSOLIDAÇÃO DAS CÓPIAS =====")
print("Linhas antes:", total_antes)
print("Linhas após consolidar cópias idênticas:", total_depois)
print("Cópias excedentes:", total_antes - total_depois)

# 2. Conferir se ficou uma linha por cliente
df_clientes_multiplas_linhas_eda = (
    df_features_sem_copias_eda
    .groupBy("id_cliente")
    .count()
    .filter(F.col("count") > 1)
)

clientes_multiplas_linhas = (
    df_clientes_multiplas_linhas_eda.count()
)

ids_nulos = (
    df_features_sem_copias_eda
    .filter(F.col("id_cliente").isNull())
    .count()
)

print("Clientes ainda com múltiplas linhas:", clientes_multiplas_linhas)
print("Linhas com id_cliente nulo:", ids_nulos)

if ids_nulos > 0:
    raise ValueError(
        "Existem IDs nulos. Revisar antes de continuar."
    )

if clientes_multiplas_linhas > 0:
    display(
        df_clientes_multiplas_linhas_eda
        .groupBy("count")
        .agg(F.count("*").alias("qtd_clientes"))
        .withColumnRenamed("count", "qtd_linhas_por_cliente")
        .orderBy("qtd_linhas_por_cliente")
    )

    raise ValueError(
        "Existem clientes com diferenças nos atributos ou metadados. "
        "Nenhuma versão foi escolhida automaticamente."
    )

# 3. Mostrar disponibilidade por cliente
print("\n===== DISPONIBILIDADE POR CLIENTE =====")

display(
    df_features_sem_copias_eda
    .groupBy("tem_endereco_disponivel")
    .agg(
        F.count("*").alias("qtd_clientes"),
        *[
            F.count(coluna).alias(f"{coluna}_preenchidas")
            for coluna in proporcoes
        ]
    )
    .orderBy("tem_endereco_disponivel")
)

# 4. Conferir nulos em todos os atributos
# Não preencher proporções ausentes com zero.
atributos = atributos_clientes + [
    "qtd_enderecos_disponiveis",
    "tem_endereco_disponivel",
    *proporcoes
]

resumo_nulos = (
    df_features_sem_copias_eda
    .agg(*[
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(coluna)
        for coluna in atributos
    ])
    .first()
    .asDict()
)

print("\n===== NULOS POR ATRIBUTO APÓS CONSOLIDAÇÃO =====")

for coluna in atributos:
    print(f"{coluna}: {resumo_nulos[coluna]}")

print("\n===== OBSERVAÇÕES =====")
print("A base resultante possui uma linha por cliente.")
print("Somente cópias iguais nas colunas da base foram consolidadas.")
print("Isso não determina a causa das repetições na origem.")
print("Domínio, tamanho do hash e idade do cadastro foram preservados.")
print("As proporções ausentes permaneceram nulas.")
print("Esta célula não cria nem modifica o rótulo experimental.")
print("A Silver e os DataFrames anteriores permanecem intactos.")
print("Nenhum dado foi gravado.")


In [0]:
# ============================================================
# RÓTULO EXPERIMENTAL — REVISÃO CADASTRAL V1
#
# Critérios preservados:
# - Falha na regra 04 da Silver (hash).
# - Falha na regra 09 da Silver (domínio).
# - Cadastro posterior à ingestão.
# ============================================================

from pyspark.sql import functions as F

chaves = ["id_cliente", "bronze_ingested_at"]
flags_rotulo = ["regra_04_falhou", "regra_09_falhou"]

# 1. Validar as colunas necessárias
for nome, df, obrigatorias in [
    (
        "clientes da Silver",
        df_clientes_ml,
        chaves + flags_rotulo
    ),
    (
        "base de atributos",
        df_features_sem_copias_eda,
        chaves + ["idade_cadastro_dias"]
    )
]:
    faltantes = [
        coluna for coluna in obrigatorias
        if coluna not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"Colunas ausentes em {nome}: {faltantes}"
        )

    if (
        df.filter(
            F.col("id_cliente").isNull()
            | F.col("bronze_ingested_at").isNull()
        )
        .limit(1)
        .count() > 0
    ):
        raise ValueError(
            f"Existem chaves nulas em {nome}."
        )

tipos = dict(df_clientes_ml.dtypes)

for coluna in flags_rotulo:
    if tipos[coluna] != "boolean":
        raise ValueError(
            f"{coluna} precisa ser boolean, "
            f"mas está como {tipos[coluna]}."
        )

# A base consolidada precisa ter uma linha por cliente.
if (
    df_features_sem_copias_eda
    .groupBy("id_cliente")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "A base de atributos ainda possui clientes repetidos."
    )

conflitos = [
    coluna for coluna in flags_rotulo
    if coluna in df_features_sem_copias_eda.columns
]

if conflitos:
    raise ValueError(
        f"A base de atributos já contém estas flags: {conflitos}. "
        "Confira a origem antes de executar a junção."
    )

# 2. Obter as flags sem multiplicar as referências
df_flags_rotulo_v1 = (
    df_clientes_ml
    .select(*chaves, *flags_rotulo)
    .distinct()
)

if (
    df_flags_rotulo_v1
    .groupBy(*chaves)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "Há flags divergentes para a mesma referência de cliente."
    )

# 3. Associar as flags à base atualizada
# Todas as colunas da base de atributos são preservadas.
df_base_rotulo_v1 = (
    df_features_sem_copias_eda
    .join(
        df_flags_rotulo_v1.withColumn(
            "_flags_encontradas",
            F.lit(True)
        ),
        on=chaves,
        how="left"
    )
)

total_origem = df_features_sem_copias_eda.count()
total_resultado = df_base_rotulo_v1.count()

if total_origem != total_resultado:
    raise ValueError(
        "A junção das flags alterou a quantidade de linhas."
    )

if total_resultado == 0:
    raise ValueError("A base está vazia.")

# Informações ausentes não devem virar rótulo zero.
if (
    df_base_rotulo_v1
    .filter(
        F.col("_flags_encontradas").isNull()
        | F.col("regra_04_falhou").isNull()
        | F.col("regra_09_falhou").isNull()
        | F.col("idade_cadastro_dias").isNull()
        | F.isnan("idade_cadastro_dias")
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "Faltam informações válidas para calcular o rótulo."
    )

# 4. Criar os motivos e o rótulo experimental
df_base_rotulada_v1 = (
    df_base_rotulo_v1
    .withColumn(
        "motivo_hash",
        F.col("regra_04_falhou")
    )
    .withColumn(
        "motivo_dominio",
        F.col("regra_09_falhou")
    )
    .withColumn(
        "motivo_cadastro_apos_ingestao",
        F.col("idade_cadastro_dias") < 0
    )
    .withColumn(
        "rotulo_revisao_v1",
        (
            F.col("motivo_hash")
            | F.col("motivo_dominio")
            | F.col("motivo_cadastro_apos_ingestao")
        ).cast("int")
    )
    .drop("_flags_encontradas")
)

# 5. Mostrar a distribuição das classes
print("===== RÓTULO EXPERIMENTAL V1 =====")
print("Clientes analisados:", total_resultado)
print("1 = sinalizado por pelo menos um dos três critérios.")
print("0 = não sinalizado por esses critérios.")

df_distribuicao_rotulo_v1 = (
    df_base_rotulada_v1
    .groupBy("rotulo_revisao_v1")
    .agg(
        F.count("*").alias("qtd_clientes")
    )
    .withColumn(
        "pct_clientes",
        F.round(
            F.col("qtd_clientes") * 100.0 / total_resultado,
            2
        )
    )
    .orderBy("rotulo_revisao_v1")
)

display(df_distribuicao_rotulo_v1)

# 6. Mostrar as combinações dos motivos
print("\n===== COMBINAÇÕES DOS MOTIVOS =====")

display(
    df_base_rotulada_v1
    .groupBy(
        "motivo_hash",
        "motivo_dominio",
        "motivo_cadastro_apos_ingestao",
        "rotulo_revisao_v1"
    )
    .agg(
        F.count("*").alias("qtd_clientes")
    )
    .orderBy(F.desc("qtd_clientes"))
)

print("\n===== OBSERVAÇÕES =====")
print("Todos os atributos da base consolidada foram preservados.")
print("Rua e coordenadas não participam do cálculo deste rótulo.")
print("As flags originais da Silver não foram recalculadas.")
print("O rótulo é experimental, derivado de regras e sem revisão humana.")
print("Sinalização não comprova fraude.")
print("Ausência de sinalização não comprova cadastro legítimo.")
print("Nenhum modelo foi treinado e nenhum dado foi gravado.")

## Conclusão da análise exploratória

A análise utilizou as tabelas ecommerce_clientes e
ecommerce_enderecos da camada Silver, sem alterar os dados de origem.

### Principais resultados

- A base de clientes apresentou 1.182 linhas e 1.135 clientes distintos.
- A consolidação de cópias idênticas na base de atributos removeu
  47 linhas excedentes, mantendo 1.135 registros.
- Foram analisados e-mails, domínios, tamanho do hash e datas.
- Foram encontrados cadastros posteriores à ingestão. O critério
  temporal precisa ser conferido com as regras utilizadas no rótulo.
- Entre as 1.135 referências de clientes, 296 tinham endereço
  disponível até o instante considerado, totalizando 323 endereços.
- As outras 839 referências não tinham endereço disponível naquele
  instante. Isso não significa que esses clientes não possuam endereço.
- A proporção de logradouros ausentes apresentou variação entre
  as referências com endereço disponível.
- As proporções de coordenadas ausentes e inválidas foram zero
  entre as referências com endereço disponível, conforme os critérios
  aplicados. Isso não comprova que a localização esteja correta.

### Atributos candidatos

Permanecem os atributos de domínio de e-mail, tamanho do usuário
do e-mail, quantidade de dígitos, tamanho do hash e idade do cadastro.

Para endereços, serão considerados quantidade e disponibilidade
na referência, além da proporção de logradouros ausentes.
Os indicadores de coordenadas permanecem documentados para avaliação
de sua utilidade.

Os atributos de CEP nulo e formato inválido foram retirados
da proposta, conforme orientação do professor.

### Cuidados para a próxima etapa

- Preservar a referência temporal ao relacionar clientes e endereços.
- Diferenciar ausência de endereço disponível de ausência de logradouro.
- Definir o tratamento das proporções nulas antes do treinamento.
- Conferir a definição do rótulo experimental com as regras da Silver.
- Tratar os sinais encontrados como indícios para revisão,
  sem interpretá-los como comprovação de fraude.

### Próxima etapa

Organizar a engenharia de features em um notebook independente,
reutilizando as transformações já desenvolvidas e lendo as tabelas
diretamente da Silver.

Nenhum modelo foi treinado nesta etapa.